# 03: Column Masking

**Exam objective:** Understand column-level masking and row-level security 
to restrict data visibility based on user groups.

**Scope of this notebook:** Column masking only. Row-level security is 
covered in 04.

**Free Edition note:** Masks can be created and attached, but testing the 
masked-vs-unmasked view requires querying as a non-owner user, which is 
impractical on Free Edition. The query results in this notebook reflect 
the owner's unmasked view; the masked behavior is documented and reasoned 
about rather than directly demonstrated.

In [0]:
-- Set up a table
USE CATALOG certprep;
USE SCHEMA governance;

-- Drop the previous table if it exists; recreate with PII to mask
DROP TABLE IF EXISTS sales_data;

CREATE TABLE sales_data (
  sale_id INT,
  region STRING,
  amount DECIMAL(10,2),
  customer_email STRING,
  customer_ssn STRING,
  sale_date DATE
);

INSERT INTO sales_data VALUES
  (1, 'North', 1250.00, 'alice@example.com', '123-45-6789', '2026-01-15'),
  (2, 'South', 890.50, 'bob@example.com',   '234-56-7890', '2026-01-16'),
  (3, 'East',  2100.75,'carol@example.com', '345-67-8901', '2026-01-17'),
  (4, 'West',  1575.25,'dan@example.com',   '456-78-9012', '2026-01-18');

SELECT * FROM sales_data;

In [0]:
-- Create UDF to mask email addresses based on group membership.
-- Members of the analysts group can view the full email.
-- Non-members see only the email domain.

CREATE OR REPLACE FUNCTION mask_email(email STRING)
RETURN
    CASE
        WHEN is_account_group_member('analysts') THEN email
        ELSE concat('***@', split(email, '@')[1])
    END;

If I, as the owner of this table, am not a member of the analysts group and query the email column while this UDF is attached to the table, I will see the masked valued.

In [0]:
-- Test the UDF directly first, before attaching it
SELECT 
  customer_email AS raw_email,
  mask_email(customer_email) AS masked_email
FROM sales_data;

In [0]:
-- Attach the UDF to the table
ALTER TABLE sales_data
ALTER COLUMN customer_email
SET MASK mask_email;

In [0]:
-- Now query the table normally. The mask is applied transparently.
SELECT * FROM sales_data;

In [0]:
-- Inspect the table's metadata to see the mask attached
DESCRIBE TABLE EXTENDED sales_data;

DESCRIBE TABLE EXTENDED will include new values to show which columns have masks and the name of the UDF on the column. 

For Example:
```
|_col_name_______|_data_type____________________________|
.........................................................
|_# Column Masks_|______________________________________|
|_customer_email_|_`certprep`.`governance`.`mask_email`_|

```

In [0]:
-- Mask SSN: show only last 4 digits for non-analysts
CREATE OR REPLACE FUNCTION mask_ssn(ssn STRING)
RETURN
  CASE
    WHEN is_account_group_member('analysts') THEN ssn
    ELSE CONCAT('XXX-XX-', RIGHT(ssn, 4))
  END;

ALTER TABLE sales_data
ALTER COLUMN customer_ssn
SET MASK mask_ssn;

In [0]:
-- Inspect the masked results
SELECT * FROM sales_data;

In [0]:
-- Drop the mask on the specified column
ALTER TABLE sales_data
ALTER COLUMN customer_email
DROP MASK;

In [0]:
-- customer_email should now show raw values; customer_ssn still masked
SELECT * FROM sales_data;

In [0]:
-- Drop the SSN mask too
ALTER TABLE sales_data
ALTER COLUMN customer_ssn
DROP MASK;

SELECT * FROM sales_data;

In [0]:
-- Drop the UDFs
DROP FUNCTION IF EXISTS mask_email;
DROP FUNCTION IF EXISTS mask_ssn;

-- Confirm
SHOW FUNCTIONS IN certprep.governance;

## Self Check Questions
1. A column mask UDF is just a SQL function. What's the practical difference between attaching a UDF as a mask versus just calling that UDF in a view (e.g., CREATE VIEW masked_sales AS SELECT mask_email(customer_email), ... FROM sales_data)? Why might a team choose one over the other?
2. The mask UDF runs on every row of every query that touches the masked column. What does this imply about how the UDF should be written? What kinds of operations inside the UDF would be a bad idea?
3. You attach a mask to a column. A user with SELECT on the table queries it and sees masked values. Same user runs DESCRIBE TABLE EXTENDED — can they see that a mask is attached, and does that matter from a security standpoint?
4. You want to mask a column differently for three different groups (full reveal for analysts, partial for support, fully redacted for everyone else). How do you structure the UDF? Is there a limit on how many group checks you'd want in one function?
5. The mask is a SQL function in the schema. What happens to the mask if the function is dropped while still attached to a column? (Predict, then look up if needed.)

## Answers
1. The practical difference of masks-on-tables vs UDFs-in-views is deciding how centralized the filter needs to be. If the mask is set on the table, there is no way of bypassing it save having the ability to drop the mask. If the UDF is used in a view, then the team has to ensure that the groups approved for masked data do not have SELECT privileges on the underlying table, and REVOKE that privilege if they do.
2. UDFs should be lightweight. Complex functions running against millions of rows results in slower performance, greater resource usage, and a rise in costs. Avoid operations like external lookups, joins, subqueries, or anything stateful.
3. The ability to see the column mask name in and of itself does not create a security risk. What actually matters is who has the privileges to alter or drop a mask. 
4. Write a CASE statement that provides WHEN conditions for the analysts and support groups respectively, followed by a broad ELSE to filter for users not in those 2 groups. There is not a limit on how many group checks you can do in a UDF, however you should aim for as few as possible. If the UDF checks 10 groups per row, that's 10 `is_account_group_member()` calls *per row*. A billion row table yields 10 billion function calls. Less is more.
5. If the function was dropped while still attached to a column in a table, querying that table would throw an error since the function cannot be located. The correct order of operations when removing masking is to detach the mask first (`ALTER TABLE ... ALTER COLUMN ... DROP MASK`), then drop the function.